In [1]:
%matplotlib widget

from __future__ import annotations

import glob
import os
import random
import time
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from dataclasses import dataclass, field
from mp_api.client import MPRester
from pathlib import Path
from pybaselines import Baseline
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.core import Composition, Structure, Lattice
from pymatgen.io.cif import CifWriter
from pymatgen.symmetry.analyzer import SymmetryUndeterminedError
from scipy.signal import find_peaks
from scipy.interpolate import interp1d
from scipy.interpolate import PchipInterpolator
from scipy.optimize import minimize
from typing import List, Tuple, Dict, Sequence


In [2]:
# Helper functions for creating simulated PXRD patterns (taken from `simulation_xrd.py`).
# -----------------------------
# Simulation (physics) config
# -----------------------------
@dataclass
class SimConfig:
    wavelength: float = 0.181
    two_theta_range: Tuple[float, float] = (1.0, 15.0)
    n_points: int = 4096

    # instrument baseline
    inst_g_deg: float = 0.010
    inst_l_deg: float = 0.008

    # TOPAS-like broadening components
    components: List[tuple] = field(default_factory=lambda: [
        ("gauss", "const", 0.0136557396),
        ("lor",   "1/cos", 0.000630507138),
        ("gauss", "tan", 0.29767628),
    ])

    # augmentation
    background_max: float = 0.0002
    noise_max: float = 0.0002

# ------------------------------------------------------------
# Peak/profile helpers
# ------------------------------------------------------------
def gaussian_profile(x: np.ndarray, center: float, fwhm: float) -> np.ndarray:
    if fwhm <= 0:
        return np.zeros_like(x)
    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    y = np.exp(-0.5 * ((x - center) / sigma) ** 2)
    s = y.sum()
    return y / (s + 1e-12)


def lorentzian_profile(x: np.ndarray, center: float, fwhm: float) -> np.ndarray:
    if fwhm <= 0:
        return np.zeros_like(x)
    gamma = fwhm / 2.0
    y = gamma**2 / ((x - center) ** 2 + gamma**2)
    s = y.sum()
    return y / (s + 1e-12)


def pseudo_voigt_profile(x: np.ndarray, center: float, fwhm_g: float, fwhm_l: float) -> np.ndarray:
    """
    Simple pseudo-Voigt approximation.
    x, center, fwhm_* are in the same angular unit (degrees 2theta here).
    """
    if fwhm_g <= 0 and fwhm_l <= 0:
        return np.zeros_like(x)

    fg = max(fwhm_g, 1e-12)
    fl = max(fwhm_l, 1e-12)

    # Approximate Voigt FWHM and mixing parameter
    f = (fl**5 + 2.69269 * fl**4 * fg + 2.42843 * fl**3 * fg**2 +
         4.47163 * fl**2 * fg**3 + 0.07842 * fl * fg**4 + fg**5) ** (1.0 / 5.0)
    r = fl / (f + 1e-12)
    eta = 1.36603 * r - 0.47719 * r**2 + 0.11116 * r**3
    eta = float(np.clip(eta, 0.0, 1.0))

    g = gaussian_profile(x, center, f)
    l = lorentzian_profile(x, center, f)
    return (1.0 - eta) * g + eta * l

# ------------------------
# Angle-dependence helpers
# ------------------------
def dep_value(theta_rad, dep_type):
    """Return the basis function for dependence at theta (radians)."""
    if dep_type == "const":
        return 1.0
    elif dep_type == "tan":
        return np.tan(theta_rad)
    elif dep_type == "1/cos":
        return 1.0 / np.cos(theta_rad)
    else:
        raise ValueError(f"Unknown dependence type: {dep_type}")
    
def build_fwhm_from_instrument_params(theta_deg, components):
    """
    Build total FWHM from components.
    components: list of tuples (shape, dependence, a)
      shape: 'gauss' or 'lor'
      dependence: 'const', 'tan', '1/cos'
      a: numbers so component = a * dependence(theta)
    Returns: (fwhm_gauss_total, fwhm_lor_total)
    """
    theta_rad = np.deg2rad(theta_deg / 2.0)  # theta = Bragg angle = tth/2 (radians)
    # note: XRD two-theta is tth; convert to theta
    # accumulate gaussian contributions in quadrature and lorentzian linearly
    gauss_sum = 0.0
    lor_sum = 0.0
    for shape, dependence, a in components:
        base = a * dep_value(theta_rad, dependence)
        # ensure non-negative
        base = max(0.0, float(base))
        if shape.lower().startswith("g"):
            gauss_sum += base
        elif shape.lower().startswith("l"):
            lor_sum += base
        else:
            raise ValueError("shape must be 'gauss' or 'lor'")
    return gauss_sum*0.85, lor_sum

def size_strain_broadening(theta_rad: np.ndarray, wavelength: float, size_nm: float, microstrain: float, K: float = 0.9):
    """
    Returns broadening terms in radians of 2theta.
    """
    L_A = size_nm * 10.0  # nm -> Angstrom
    beta_size = (K * wavelength) / (L_A * np.cos(theta_rad) + 1e-12)   # Lorentzian-like
    beta_strain = 4.0 * microstrain * np.tan(theta_rad)                  # Gaussian-like
    return np.rad2deg(beta_size), np.rad2deg(beta_strain)


# ------------------------------------------------------------
# CIF -> full pattern
# ------------------------------------------------------------
def simulate_pattern_from_cif(
    cif_path: str,
    cfg: SimConfig,
    size_nm: float,
    microstrain: float,
    scale: float = 1.0,
    lattice_dims: tuple[float, float, float] = (0, 0, 0)
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Simulate a single-phase full XRD pattern from CIF.
    """
    structure = Structure.from_file(cif_path)

    # If lattice sizes are provided, then provide the structure with the new lattuce.
    if(lattice_dims != (0, 0, 0)):
        old_angles = structure.lattice.angles
        new_lattice = Lattice.from_parameters(a = lattice_dims[0], b = lattice_dims[1], c = lattice_dims[2], alpha = old_angles[0], beta = old_angles[1], gamma = old_angles[2])
        structure.lattice = new_lattice

    xrd = XRDCalculator(wavelength=cfg.wavelength, symprec=0.1)
    pat = xrd.get_pattern(structure, two_theta_range=cfg.two_theta_range)

    x = np.linspace(cfg.two_theta_range[0], cfg.two_theta_range[1], cfg.n_points)
    y = np.zeros_like(x, dtype=np.float64)

    # if no components specified, usefixed small instrument broadening, converted to radians then back to degrees later
    if cfg.components is None:
        beta_g_inst_fix = cfg.inst_g_deg
        beta_l_inst_fix = cfg.inst_l_deg

    else:
        components = cfg.components

    for tth, inten in zip(pat.x, pat.y):
        if inten <= 0:
            continue

        theta_rad = np.deg2rad(tth / 2.0)
        beta_size, beta_strain = size_strain_broadening(
            theta_rad=theta_rad,
            wavelength=cfg.wavelength,
            size_nm=size_nm,
            microstrain=microstrain,
        )
        # compute FWHM at this peak (two_theta units)
        beta_g_inst, beta_l_inst = (beta_g_inst_fix, beta_l_inst_fix) if (cfg.components is None) else build_fwhm_from_instrument_params(tth, components)
        # Combine widths in degrees of 2theta
        fwhm_g_deg = np.sqrt(beta_g_inst**2 + beta_strain**2)
        fwhm_l_deg = beta_l_inst + beta_size
        profile = pseudo_voigt_profile(x, tth, fwhm_g_deg, fwhm_l_deg)
        y += scale * inten * profile

    # Normalize to unit max so weights mean relative contribution
    y = y - y.min()
    if y.max() > 0:
        y = y / y.max()

    return x.astype(np.float32), y.astype(np.float32)


# ------------------------------------------------------------
# Mixture weight sampling
# ------------------------------------------------------------
def sample_mixture_weights(
    n_phases: int,
    dominant_range: Tuple[float, float] = (0.450, 0.70),
    minor_alpha: float = 0.35,
) -> np.ndarray:
    """
    Sample weights so that one phase is dominant and the rest are weak.
    Returns weights that sum to 1.
    """
    if n_phases <= 0:
        raise ValueError("n_phases must be >= 1")

    if n_phases == 1:
        return np.array([1.0], dtype=np.float32)

    dominant_idx = np.random.randint(n_phases)
    dominant_w = np.random.uniform(*dominant_range)
    remaining = 1.0 - dominant_w

    # Dirichlet for the minor phases
    minor = np.random.dirichlet(alpha=np.full(n_phases - 1, minor_alpha))
    minor = minor * remaining

    weights = np.zeros(n_phases, dtype=np.float32)
    weights[dominant_idx] = dominant_w
    weights[np.arange(n_phases) != dominant_idx] = minor
    weights = weights / weights.sum()
    return weights


# ------------------------------------------------------------
# Build one synthetic mixture
# ------------------------------------------------------------
def generate_mixture_pattern(
    cif_paths: Sequence[str],
    cfg: SimConfig,
    n_phases_range: Tuple[int, int] = (2, 4),
    allow_replacement: bool = False
) -> Dict:
    """
    Generate one mixed XRD pattern from multiple CIFs.

    Returns a dict with:
      x, y, chosen_cifs, weights, size_nm, microstrain
    """
    cif_paths = list(cif_paths)
    if len(cif_paths) == 0:
        raise ValueError("cif_paths is empty")

    n_min, n_max = n_phases_range
    n_phases = random.randint(n_min, n_max)
    n_phases = min(n_phases, len(cif_paths)) if not allow_replacement else n_phases

    chosen = random.sample(cif_paths, n_phases) if not allow_replacement else random.choices(cif_paths, k=n_phases)

    weights = sample_mixture_weights(n_phases)

    x = np.linspace(cfg.two_theta_range[0], cfg.two_theta_range[1], cfg.n_points)
    y_mix = np.zeros_like(x, dtype=np.float64)

    phase_meta = []

    for cif_path, w in zip(chosen, weights):
        # Per-phase variability
        size_nm = np.random.uniform(20.0, 200.0)
        microstrain = np.random.uniform(0.0, 0.002)
        x_i, y_i = simulate_pattern_from_cif(
            cif_path=cif_path,
            cfg=cfg,
            size_nm=size_nm,
            microstrain=microstrain,
            scale=1.0
        )

        # x_i should match x if cfg is fixed; kept here for clarity
        y_mix += float(w) * y_i

        phase_meta.append({
            "cif": os.path.basename(cif_path),
            "weight": float(w),
            "size_nm": float(size_nm),
            "microstrain": float(microstrain),
        })

    # Add background
    bg_level = np.random.uniform(0.0, cfg.background_max)
    bg = bg_level * (1.0 + 0.02 * (x - x.mean()) + 0.0005 * (x - x.mean()) ** 2)
    y_mix += bg

    # Add noise
    noise_level = np.random.uniform(0.0, cfg.noise_max)
    y_mix += np.random.normal(0.0, noise_level * max(y_mix.max(), 1e-12), size=y_mix.shape)

    # Normalize
    y_mix = y_mix - y_mix.min()
    if y_mix.max() > 0:
        y_mix = y_mix / y_mix.max()
    
    return {
        "x": x.astype(np.float32),
        "y": y_mix.astype(np.float32),
        "chosen_cifs": chosen,
        "weights": weights.astype(np.float32),
        "phases": phase_meta,
    }


# ------------------------------------------------------------
# Batch generator
# ------------------------------------------------------------
def generate_mixture_batch(
    cif_paths: Sequence[str],
    cfg: SimConfig,
    n_samples: int = 100,
    n_phases_range: Tuple[int, int] = (2, 4),
) -> List[Dict]:
    """
    Generate a batch of synthetic mixture patterns.
    """
    batch = []
    for _ in range(n_samples):
        sample = generate_mixture_pattern(
            cif_paths=cif_paths,
            cfg=cfg,
            n_phases_range=n_phases_range,
            allow_replacement=False
        )
        batch.append(sample)
    return batch

In [3]:
# Pulled from `materials_project_query.py`. Contains helper functions to request information from the Materials Project database using their API.
def save_structure_to_cif(structure, output_file):
    """This function is a helper function to save the cif files.

    Parameters
    ----------
    structure : pymatgen.core.structure.Structure
        The structure to save as a CIF file.
    file : os.PathLike
        The path to save the structure to.
    """

    output_directory = os.path.dirname(output_file)
    os.makedirs(output_directory, exist_ok=True)
    cif_writer = CifWriter(structure, symprec=0.1)
    cif_writer.write_file(output_file)


def pull_data_from_Materials_Project(
    output_directory,
    formulas=[],
    fields=["material_id", "structure", "formula_pretty"],
    api_key=os.environ["PMG_API_KEY"]
):
    """Executes a query to the Materials Project, and saves those materials
    to disk as CIF files.
    
    Parameters
    ----------
    output_directory : os.PathLike
        Path to the directory to save the resultant CIF files.
    elements : list
        Elements to include in the query.
    chemsys : str
        The type of chemical system to pull. For example, "Pb-S-*".
    fields : list, optional
        The fields to keep from the Materials Project.
    api_key : str, optional
        Materials Project API key. Defaults to an environment variable
        "PMG_API_KEY".
    """
    for formula in formulas:
        with MPRester(api_key=api_key) as mpr:
            # took out chemsys=chemsys from params
            docs = mpr.materials.search(
                formula=formula,
                fields=fields
            )

        for d in docs:
            output_file = Path(output_directory) / f"{d.material_id}_structure.cif"
            save_structure_to_cif(d.structure, output_file)

In [4]:
# Declare some constants for use with pattern simulation.
cfg = SimConfig(
    wavelength = 0.1867,
    two_theta_range = (0.5, 15.0),
    n_points = 4096,
    )
size_nm = 380
microstrain = 0.001

In [5]:
# Given a file path to a CIF file `cif_file`, simulates a PXRD pattern and saves and returns the resulting data.
# If `lattice_sizes` are provided, simulates the PXRD pattern as if the `cif_file` had these lattice sizes. Otherwise, simulates using the provided CIF sizes.
# If `append` is provided, sticks it to the end of the file name for easier file organization. Otherwise, nothing is appended to this new file.
# Returns a 2D Numpy array containing the resulting data of the simulated graph (col. 1: x-data, col. 2: y-data).
def save_simulated_pattern(cif_file: str, cif_directory: str = "", lattice_sizes: tuple[float, float, float] = (0, 0, 0), append: str = None) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter(action = "ignore")
        # Create the simulated pattern from the given parameters.
        pattern = simulate_pattern_from_cif(cif_path = cif_file, cfg = cfg, size_nm = size_nm, microstrain = microstrain, lattice_dims = lattice_sizes)
        # Consolidate the pattern results to extract the proper data.
        pattern_data = np.column_stack(tup = (pattern[0], pattern[1]))
        # Save the pattern's data to the graphs directory (elided right now to avoid bloat in my own directories).
        np.savetxt(fname = os.path.join(cif_directory, os.path.splitext(os.path.basename(cif_file))[0] + (append or "") + "_sim.chi"), X = pattern_data, delimiter = ' ')
        # Return the pattern's data for further use.
        return pattern_data

In [6]:
# Given a file path to a XY file `dat_file`, simulates and returns the given experimental data.
# Returns a 2D Numpy array containing the resulting data of the experimental graph (col. 1: x-data, col. 2: y-data).
def extract_experimental_pattern(dat_file: str, skiprows: int) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.simplefilter(action = "ignore")
        # Extract the experimental data from the given file.
        x_data, y_data = np.loadtxt(fname = dat_file, unpack = True, skiprows = skiprows)
        # Consolidate the pattern results to extract the proper data.
        exp_data = np.column_stack(tup = (x_data, y_data))
        # Return the pattern's data for further use.
        return exp_data

In [7]:
# Extracts a baseline from a given experimental pattern `mystery_pattern` and returns that same baseline.
def extract_baseline(mystery_pattern: np.ndarray):
    # Create a range for the baseline based on the x-data.
    baseline_fitter = Baseline(x_data = mystery_pattern[:, 0])
    # Create an official baseline based on the y-data.
    baseline, params = baseline_fitter.asls(data = mystery_pattern[:, 1], lam = 1e6, p = 0.01)
    # Return the calculated value (`params` isn't relevant).
    return baseline

In [8]:
# Extract and return the lattice from the given CIF file `cif_path`.
def get_lattice(cif_path: str) -> Lattice:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Extract the structure from the given CIF file.
        structure = Structure.from_file(filename = cif_path)
        # Return the lattice of the obtained structure.
        return structure.lattice

In [9]:
# Calculates the roughly optimal lattice size for the CIF `base_pattern` when it appears in the mixture `target_pattern`.
# Returns a tuple of the form ([lattice-sizes], [correlation-to-target]).
def find_latlengths(target_pattern: np.ndarray, base_pattern: str, cif_directory: str = "") -> List[Tuple[float, float]]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Obtain the lattice from the given candidate CIF file.
        lattice_sizes = get_lattice(base_pattern).abc
        # Normalize the experimental data to be in line with the simulated data (scaling with the greatest dataset's value).
        base_sim = save_simulated_pattern(cif_file = base_pattern, cif_directory = cif_directory, lattice_sizes = lattice_sizes, append = "base")
        pchip_interpolator = PchipInterpolator(base_sim[:, 0], base_sim[:, 1])
        y_interpolated = pchip_interpolator(target_pattern[:, 0])
        sim_peak_locations = find_peaks(y_interpolated, height = 0.01)[0]
        y_experimental = target_pattern[:, 1]
        scale_aggregate = 0
        for peak_index in sim_peak_locations:
            scale_factor = y_interpolated[peak_index] / y_experimental[peak_index]
            scale_aggregate = scale_aggregate + scale_factor
        scale_aggregate = scale_aggregate / len(sim_peak_locations)
        target_pattern[:, 1] = target_pattern[:, 1] * scale_aggregate
        # Extract the baseline from this mystery pattern.
        baseline = extract_baseline(mystery_pattern = target_pattern)
        # Calculate the correlations between the target CIF and the simulated CIFs of various lattice sizes.
        lattices_and_correlations = []
        # Define the function to minimize for Nelder-Mead Minimization.
        def objective_function(x: float,  original_params: tuple[float, float, float], fix_lattice: tuple[bool, bool, bool] = (False, False, False)):
            x = x[0]
            lattice_sizes = []
            for i in range(len(fix_lattice)):
                if(fix_lattice[i] == True):
                    lattice_sizes.append(original_params[i])
                else:
                    lattice_sizes.append(x)
            # Create a simulated pattern with specific lattice size.
            sim_pattern = save_simulated_pattern(cif_file = base_pattern, cif_directory = cif_directory, lattice_sizes = lattice_sizes, append = str(x))
            # Interpolate the data of the simulated pattern to that of the target pattern.
            pchip_interpolator = PchipInterpolator(sim_pattern[:, 0], sim_pattern[:, 1])
            y_interpolated = pchip_interpolator(target_pattern[:, 0])
            y_interpolated = y_interpolated + baseline
            # Add the interpolated data to the OLS model.
            sim_data = sm.add_constant(y_interpolated)
            # Fit the OLS model and extract the correlation data from the results.
            model_results = sm.OLS(target_pattern[:, 1], sim_data).fit()
            correlation = model_results.params
            # Reset the OLS model from further use.
            model_results.remove_data()
            # Append the lattice size and corresponding correlation to the aggregate data.
            lattices_and_correlations.append((x, correlation[1]))
            # Return the negative correlation (necessary for finding the maximum as SciPy doesn't have a maximize method).
            return -1 * correlation[1]
        # Extract the lattice from the given CIF pattern.
        base_lattice = get_lattice(base_pattern).abc
        cubic_candidate = base_lattice[0] == base_lattice[1] and base_lattice[0] == base_lattice[2]
        max_iter = 2
        # Run the candidate function multiple times to converge onto a minimum for each lattice parameter, and normalize the results afterwards.
        # If there is a cubic candidate, then some of the calculation can be elided.
        if(cubic_candidate):
            initial = base_lattice[0]
            candidates = []
            for _ in range(max_iter):
                minimize(fun = objective_function, x0 = initial, method = "Nelder-Mead", args = (base_lattice, (False, False, False)))
                initial = max(initial, min(initial * 1.01, lattices_and_correlations[-1][0]))
                candidates.append(lattices_and_correlations[-1])
            numerator = 0
            denominator = 0
            for result in candidates:
                numerator = numerator + result[0] * result[1]
                denominator = denominator + result[1]
            true_result = numerator / denominator
            return (true_result, true_result, true_result)
        # If the candidate is not cubic, each of the lattice parameters must be calculated individually.
        else:
            return_result = [0, 0, 0]
            for i in range(3):
                fixes = [(False, True, True), (True, False, True), (True, True, False)]
                initial = base_lattice[i]
                candidates = []
                for _ in range(max_iter):
                    minimize(fun = objective_function, x0 = initial, method = "Nelder-Mead", args = (base_lattice, fixes[i]))
                    initial = max(initial, min(initial * 1.01, lattices_and_correlations[-1][0]))
                    candidates.append(lattices_and_correlations[-1])
                # Calculate the average result from the calculated minima.
                numerator = 0
                denominator = 0
                for result in candidates:
                    numerator = numerator + result[0] * result[1]
                    denominator = denominator + result[1]
                true_result = numerator / denominator
                return_result[i] = true_result
            # Return the estimated lattice size.
            return tuple(return_result)

In [10]:
# Calculate the correlation between the mixture data `mixture_data` and every CIF in `cif_list` individually.
def calculate_highest_correlation(mixture_data, cif_list: list[str], cif_directory: str = "") -> np.ndarray:
    # Create a container for holding the predictions for each component.
    mixture_correlations = []
    # For each CIF file given in the file list:
    for i in range(len(cif_list)):
        # Obtain the component data from the CIF.
        try:
            component_data = save_simulated_pattern(cif_list[i], cif_directory = cif_directory)
        except SymmetryUndeterminedError:
            print("symmetry error")
            mixture_correlations.append(0)
            continue
        # Create an interpolator and use it to estimate the CIF's data on the mixture's x-axis.
        pchip_interpolator = PchipInterpolator(component_data[:, 0], component_data[:, 1])
        y_interpolated = pchip_interpolator(mixture_data[:, 0])
        y_experimental = mixture_data[:, 1]
        # Add the interpolated data to the OLS model.
        sim_model = sm.add_constant(y_interpolated)
        # Fit the OLS model and extract the correlation data from the results.
        model_results = sm.OLS(y_experimental, sim_model).fit()
        correlation = model_results.params
        mixture_correlations.append(correlation[1])
        # Reset the OLS model from further use.
        model_results.remove_data()
    # Normalize the correlation data to more accurately reflect actual weights.
    mixture_correlations = np.clip(mixture_correlations, a_min = 0, a_max = None)
    mixture_correlations = mixture_correlations / np.sum(mixture_correlations)
    mixture_correlations = mixture_correlations * 100
    return mixture_correlations.tolist()

In [12]:
# Queries the Materials Science API for candidate phase CIFs and calculates the top 10 correlations for each experimental file.
# `data_directory` is a string depicting the directory from which to take the experimental data from. The program will read every file with the ".xy" extension in that directory
#  and process it.
# `cif_directory` is a string depicting the directory to which to save the CIFs taken from the Materials Science API.
# `formulas` is a list of strings, with each element being a string of a compound to search for in the MS API.
# `skiprows` is an integer denoting how many rows to skip in the experimental data before reading coordinates. Use this to skip things like header lines, which will otherwise
#  cause an error.
# Return a dataframe with every applicable CIF from the MS API and their correlation to each data file.
def phase_score_ranking(data_directory: str = "", cif_directory: str = "", formulas: List[str] = [], skiprows: int = 0):
    # The driver for the above cells. Creates a dataset of three-phase mixtures and calculates the shifted and unshifted correlation for them.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # Pull data from the Materials Project database using their API.
        pull_data_from_Materials_Project(output_directory = cif_directory, formulas = formulas)
        # Extract the CIF files from the proper directory and put them into a list.
        cif_list = glob.glob(cif_directory + "*.cif")
        # Create a container for the aggregate results.
        aggregate_mixture_data = []
        # Extract the experimental files from the correct location.
        exp_data_list = glob.glob(data_directory + "*.xy")
        trimmed_exp_list = []
        # Create labels for another axis of the resulting dataframe.
        for exp in exp_data_list:
            trimmed_exp_list.append(os.path.basename(exp))
        print(trimmed_exp_list)
        # For each file listed in the file list:
        for i in range(len(exp_data_list)):
            print("Phase scores for " + trimmed_exp_list[i] + ":")
            # Extract the data from the experimental pattern (`skiprows` is used for skipping the header).
            mixture_data = extract_experimental_pattern(exp_data_list[i], skiprows = skiprows)
            # Find the correlation data between the mixtures and the component CIFs (shifted).
            results = calculate_highest_correlation(mixture_data = mixture_data, cif_list = cif_list, cif_directory = cif_directory)
            # Add this correlation to the aggregate data container.
            aggregate_mixture_data.append(results)
            for _ in range(10):
                max_index = np.argmax(results)
                if(results[max_index] == 0):
                    break
                chemical_formula = ""
                with open(cif_list[max_index]) as f:
                    f.readline().strip('\n')
                    chemical_formula = f.readline().strip('\n')
                print("   - " + str(results[max_index]) + ": " + chemical_formula + " " + str(get_lattice(cif_list[max_index]).abc))
                results = np.delete(results, max_index)
        # Create a big dataframe from all of the mixture data.
        final_frame = pd.DataFrame(aggregate_mixture_data)
        final_frame.columns = [os.path.basename(cif) for cif in cif_list]
        final_frame.index = trimmed_exp_list
        # Return the final results.
        return final_frame

In [13]:
# Run the phase score ranking for preset data. Parameters can be changed directly here.
# An example call is provided for a spinel sintering dataset. This call can be modified to analyze whatever dataset you want.
# results = phase_score_ranking(data_directory = "../data/data_spinel/",cif_directory = "../data/cifs_misc/materials_spinel_cifs/",  formulas = ["Al2O3", "MgO", "ZrO2", "ZrO", "MgAl2O4"], skiprows = 6)